# ⚖️ Fine-Tuning Phi-3-mini with QLoRA for Legal Provision Classification
### End-to-End Google Colab T4 Training Pipeline · LexGLUE/LEDGAR Benchmark

- **Base Model:** `microsoft/Phi-3-mini-4k-instruct` (3.8B parameters)
- **Technique:** QLoRA (4-bit NF4 Quantization + LoRA rank-16 adapters)
- **Dataset:** LexGLUE LEDGAR (100 legal contract clause categories)
- **Loss Function:** Class-Weighted Cross-Entropy (handling 137.7x class disparity)
- **Orchestration:** TRL `SFTTrainer` with completion token loss masking
- **Checkpointing:** Direct Hugging Face Hub push (`push_to_hub=True`)
- **Experiment Tracking:** Weights & Biases (`wandb`)

## 1. Verify GPU Allocation
Google Colab Free Tier provides an NVIDIA T4 GPU (16GB VRAM). Let's confirm GPU presence.

In [ ]:
!nvidia-smi

## 2. Install Required Dependencies
We install pinned, battle-tested versions optimized for 4-bit QLoRA and TRL SFT.

In [ ]:
!pip install -q \
    "torch>=2.3.0" \
    "transformers>=4.44.0" \
    "peft>=0.12.0" \
    "bitsandbytes>=0.43.3" \
    "trl>=0.9.6" \
    "datasets>=2.21.0" \
    "accelerate>=0.33.0" \
    "scikit-learn>=1.5.0" \
    "wandb>=0.17.5" \
    "scipy>=1.13.0" \
    "einops>=0.8.0"

## 3. Authenticate with Hugging Face & Weights & Biases
> [!TIP]
> **Recommended:** Add `HF_TOKEN` and `WANDB_API_KEY` to Colab's Secrets tab (the key icon 🔑 on the left sidebar) with notebook access enabled.
> If not using Secrets, you will be prompted interactively.

In [ ]:
import os

import wandb
from huggingface_hub import login

# Attempt loading from Colab userdata / Secrets
try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
    wandb_key = os.environ.get("WANDB_API_KEY")

# Hugging Face Authentication
if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    print("Logged into Hugging Face Hub successfully.")
else:
    print("HF_TOKEN not found in Secrets. Prompting login...")
    login()

# Weights & Biases Authentication
if wandb_key:
    wandb.login(key=wandb_key)
    print("Logged into Weights & Biases successfully.")
else:
    print("WANDB_API_KEY not found in Secrets. Prompting login...")
    wandb.login()

## 4. Hyperparameters & Repository Configuration
Configure your target repository name on Hugging Face Hub.

In [ ]:
import torch

# =====================================================================
# USER CONFIGURATION: Set your Hugging Face Hub username & repo name
# =====================================================================
HF_USERNAME = "your-username"  # <-- Change this to your Hugging Face username
HUB_REPO_NAME = "phi3-legal-clause-qlora"
HUB_MODEL_ID = f"{HF_USERNAME}/{HUB_REPO_NAME}"

BASE_MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
OUTPUT_DIR = "./outputs/phi3-ledgar-qlora"

# Training Hyperparameters (T4 16GB Optimized)
NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch size = 4 * 8 = 32
LEARNING_RATE = 2.0e-4
MAX_SEQ_LENGTH = 512
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
SEED = 42

print(f"Target Hugging Face Hub Repo: {HUB_MODEL_ID}")
print(f"Effective Batch Size: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

## 5. Dataset Loading & Balanced Class Weight Computation
Load the LexGLUE LEDGAR benchmark (60,000 train, 10,000 val, 10,000 test) across 100 contract provision categories.
Compute normalized inverse class frequency weights:
$$w_c = \frac{N}{C \cdot N_c}$$
to balance the 137.7x class disparity during instruction tuning.

In [ ]:
import numpy as np
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight

print("Loading 'coastalcph/lex_glue' (subset: 'ledgar')...")
raw_dataset = load_dataset("coastalcph/lex_glue", "ledgar")

label_names = raw_dataset["train"].features["label"].names
num_classes = len(label_names)
print(f"Loaded {num_classes} classes. Splits: { {k: len(raw_dataset[k]) for k in raw_dataset} }")

# Compute balanced class weights from training labels
train_labels = np.asarray(raw_dataset["train"]["label"], dtype=np.int64)
raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_labels,
).astype(np.float32)

# Clip extreme tail weights at 15.0 to prevent gradient spikes, and normalize mean to 1.0
clipped_weights = np.clip(raw_weights, a_min=None, a_max=15.0)
normalized_weights = clipped_weights / np.mean(clipped_weights)
class_weights_tensor = torch.tensor(normalized_weights, dtype=torch.float32).cuda()

print(
    f"Class weights computed: min={class_weights_tensor.min():.4f}, max={class_weights_tensor.max():.4f}, mean={class_weights_tensor.mean():.4f}"
)

## 6. Prompt Formatting for Causal Instruction Tuning
Convert provisions into Phi-3 conversational format (`<|user|>` $\rightarrow$ `<|assistant|>`).

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert legal AI assistant. Classify the following contract provision into "
    "exactly one of the 100 standard LEDGAR provision categories. Respond with only the exact category name."
)


def format_example(example):
    cleaned_clause = example["text"].strip()
    category = label_names[example["label"]]
    prompt = (
        f"<|user|>\n"
        f"{SYSTEM_PROMPT}\n\n"
        f'Contract Provision:\n"{cleaned_clause}"\n<|end|>\n'
        f"<|assistant|>\n"
        f"{category}<|end|>"
    )
    return {"text": prompt, "class_id": example["label"]}


print("Formatting train and validation partitions...")
train_dataset = raw_dataset["train"].map(format_example, desc="Formatting train")
val_dataset = raw_dataset["validation"].map(format_example, desc="Formatting val")

print("\n--- Formatted Prompt Preview ---")
print(train_dataset[0]["text"][:350] + "...")

## 7. Load Tokenizer & 4-bit Quantized Phi-3-mini Model
- 4-bit NormalFloat (NF4) quantization via `bitsandbytes`
- Double quantization enabled (saves ~0.37 bits/parameter)
- `bfloat16` compute dtype
- Target LoRA modules: `o_proj`, `qkv_proj`, `gate_up_proj`, `down_proj`

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Fix RoPE scaling for Phi-3-mini-4k (4k variant uses standard RoPE without scaling)
config = AutoConfig.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
config.rope_scaling = None

# 3. BitsAndBytes 4-bit NF4 Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 4. Load Base Model
print(f"Loading {BASE_MODEL_ID} in 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
)

# 5. Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Custom Weighted SFTTrainer & Completion-Only Collator
- `DataCollatorForCompletionOnlyLM`: Sets labels to `-100` on prompt tokens so loss is computed **strictly on the generated category name**.
- `WeightedSFTTrainer`: Custom loss computation that scales each sample's completion loss by its class weight $w_{c_i}$.

In [ ]:
import torch
import torch.nn as nn
from transformers import DataCollatorForSeq2Seq, Trainer

# 1. Explicitly tokenize and mask prompt tokens (removing raw string columns)
response_token_ids = tokenizer.encode("<|assistant|>\n", add_special_tokens=False)


def tokenize_and_mask(example):
    cleaned_clause = example["text"].strip()
    category = label_names[example["label"]]
    full_prompt = (
        f"<|user|>\n"
        f"{SYSTEM_PROMPT}\n\n"
        f'Contract Provision:\n"{cleaned_clause}"\n<|end|>\n'
        f"<|assistant|>\n"
        f"{category}<|end|>"
    )
    enc = tokenizer(
        full_prompt,
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding=False,
    )
    input_ids = enc["input_ids"]
    labels = list(input_ids)

    r_len = len(response_token_ids)
    start_idx = -1
    for j in range(len(input_ids) - r_len + 1):
        if input_ids[j : j + r_len] == response_token_ids:
            start_idx = j + r_len
            break
    if start_idx != -1:
        labels[:start_idx] = [-100] * start_idx

    return {
        "input_ids": input_ids,
        "attention_mask": enc["attention_mask"],
        "labels": labels,
        "class_id": example["label"],
    }


print("Tokenizing dataset and building completion labels...")
train_tokenized = raw_dataset["train"].map(
    tokenize_and_mask,
    remove_columns=raw_dataset["train"].column_names,
    desc="Tokenizing train",
)
val_tokenized = raw_dataset["validation"].map(
    tokenize_and_mask,
    remove_columns=raw_dataset["validation"].column_names,
    desc="Tokenizing val",
)


# 2. Collator that pads numeric tensors and packages class_id
class WeightedDataCollator(DataCollatorForSeq2Seq):
    def __call__(self, features):
        class_ids = [f.pop("class_id") for f in features]
        batch = super().__call__(features)
        batch["class_id"] = torch.tensor(class_ids, dtype=torch.long)
        return batch


collator = WeightedDataCollator(tokenizer=tokenizer, pad_to_multiple_of=8)


# 3. Custom Weighted Trainer
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        class_ids = inputs.pop("class_id", None)
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        batch_size = shift_labels.size(0)
        vocab_size = shift_logits.size(-1)

        loss_fct = nn.CrossEntropyLoss(reduction="none")
        token_loss = loss_fct(
            shift_logits.view(-1, vocab_size),
            shift_labels.view(-1),
        ).view(batch_size, -1)

        mask = (shift_labels != -100).float()
        token_counts = mask.sum(dim=1).clamp(min=1.0)
        sample_loss = (token_loss * mask).sum(dim=1) / token_counts

        if class_ids is not None and self.class_weights is not None:
            dev = sample_loss.device
            sample_w = self.class_weights.to(dev)[class_ids.to(dev)]
            loss = (sample_loss * sample_w).sum() / sample_w.sum().clamp(min=1e-8)
        else:
            loss = sample_loss.mean()

        return (loss, outputs) if return_outputs else loss


print("WeightedTrainer and WeightedDataCollator ready!")

## 9. Training Configuration & Launch
Setting up `TrainingArguments` with:
- `push_to_hub=True` to checkpoint directly to Hugging Face Hub
- `optim="paged_adamw_8bit"` to prevent memory spikes
- `report_to="wandb"` for real-time loss tracking

In [ ]:
import torch
import torch.nn as nn
from transformers import DataCollatorForSeq2Seq, Trainer

# 1. Explicitly tokenize and mask prompt tokens (removing raw string columns)
response_token_ids = tokenizer.encode("<|assistant|>\n", add_special_tokens=False)


def tokenize_and_mask(example):
    cleaned_clause = example["text"].strip()
    category = label_names[example["label"]]
    full_prompt = (
        f"<|user|>\n"
        f"{SYSTEM_PROMPT}\n\n"
        f'Contract Provision:\n"{cleaned_clause}"\n<|end|>\n'
        f"<|assistant|>\n"
        f"{category}<|end|>"
    )
    enc = tokenizer(
        full_prompt,
        max_length=MAX_SEQ_LENGTH,
        truncation=True,
        padding=False,
    )
    input_ids = enc["input_ids"]
    labels = list(input_ids)

    r_len = len(response_token_ids)
    start_idx = -1
    for j in range(len(input_ids) - r_len + 1):
        if input_ids[j : j + r_len] == response_token_ids:
            start_idx = j + r_len
            break
    if start_idx != -1:
        labels[:start_idx] = [-100] * start_idx

    return {
        "input_ids": input_ids,
        "attention_mask": enc["attention_mask"],
        "labels": labels,
        "class_id": example["label"],
    }


print("Tokenizing dataset and building completion labels...")
train_tokenized = raw_dataset["train"].map(
    tokenize_and_mask,
    remove_columns=raw_dataset["train"].column_names,
    desc="Tokenizing train",
)
val_tokenized = raw_dataset["validation"].map(
    tokenize_and_mask,
    remove_columns=raw_dataset["validation"].column_names,
    desc="Tokenizing val",
)


# 2. Collator that pads numeric tensors and packages class_id
class WeightedDataCollator(DataCollatorForSeq2Seq):
    def __call__(self, features):
        class_ids = [f.pop("class_id") for f in features]
        batch = super().__call__(features)
        batch["class_id"] = torch.tensor(class_ids, dtype=torch.long)
        return batch


collator = WeightedDataCollator(tokenizer=tokenizer, pad_to_multiple_of=8)


# 3. Custom Weighted Trainer
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        class_ids = inputs.pop("class_id", None)
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        batch_size = shift_labels.size(0)
        vocab_size = shift_logits.size(-1)

        loss_fct = nn.CrossEntropyLoss(reduction="none")
        token_loss = loss_fct(
            shift_logits.view(-1, vocab_size),
            shift_labels.view(-1),
        ).view(batch_size, -1)

        mask = (shift_labels != -100).float()
        token_counts = mask.sum(dim=1).clamp(min=1.0)
        sample_loss = (token_loss * mask).sum(dim=1) / token_counts

        if class_ids is not None and self.class_weights is not None:
            dev = sample_loss.device
            sample_w = self.class_weights.to(dev)[class_ids.to(dev)]
            loss = (sample_loss * sample_w).sum() / sample_w.sum().clamp(min=1e-8)
        else:
            loss = sample_loss.mean()

        return (loss, outputs) if return_outputs else loss


print("WeightedTrainer and WeightedDataCollator ready!")

## 10. Push Adapter & Tokenizer to Hugging Face Hub
Upload the trained LoRA adapter (~100MB) and configured tokenizer to your Hugging Face repository.

In [ ]:
print(f"Pushing final best model adapter to Hugging Face Hub: {HUB_MODEL_ID}...")
trainer.push_to_hub(commit_message="Fine-tuned Phi-3-mini QLoRA on LexGLUE/LEDGAR")
tokenizer.push_to_hub(HUB_MODEL_ID)

print("\n" + "=" * 70)
print(f"🎉 TRAINING COMPLETE! Adapter published at: https://huggingface.co/{HUB_MODEL_ID}")
print("=" * 70)

## 11. Quick Generation Test on a Sample Provision
Test that the trained adapter generates clean, valid LEDGAR category names.

In [ ]:
test_sample = raw_dataset["test"][0]
test_clause = test_sample["text"]
expected_category = label_names[test_sample["label"]]

test_prompt = (
    f'<|user|>\n{SYSTEM_PROMPT}\n\nContract Provision:\n"{test_clause}"\n<|end|>\n<|assistant|>\n'
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
)
print("Test Provision:\n", test_clause[:200] + "...")
print(f"\nExpected Category : {expected_category}")
print(f"Generated Output  : {generated_response.strip()}")